# Gas Prices — Data Preparation
Converts yearly xlsx files into a single cleaned CSV with dates as index and cities as columns.

**Folder structure:**
```
data/
  raw/        ← all xlsx source files
  processed/  ← output CSV (created automatically)
```

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime

BASE_DIR         = Path().resolve()
raw_folder       = BASE_DIR / 'data' / 'raw'
processed_folder = BASE_DIR / 'data' / 'processed'

processed_folder.mkdir(parents=True, exist_ok=True)

print(f'Raw       : {raw_folder} (exists: {raw_folder.exists()})')
print(f'Processed : {processed_folder} (exists: {processed_folder.exists()})')

## Step 1 — Helper functions

Three date formats exist across the source files:
- **2000**: Excel serial numbers (e.g. `46026`)
- **2001–2014**: datetime objects with wrong year (always shows 2026)
- **2015–2026**: date strings like `1/7`

Two layout formats:
- **2015–2016**: dates in row 0, cities start at row 1
- **All others**: dates in row 2, cities start at row 3

In [ ]:
def parse_dates(date_row, year):
    """Handle all three date formats, always forcing the correct year."""
    parsed = []
    for val in date_row:
        try:
            if pd.isna(val):
                parsed.append(pd.NaT)
            elif isinstance(val, (int, float)):
                # Excel serial number → real date
                d = pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(val))
                parsed.append(d.replace(year=int(year)))
            elif hasattr(val, 'month'):
                # datetime object with wrong year
                parsed.append(val.replace(year=int(year)))
            else:
                # String like '1/7'
                parsed.append(pd.to_datetime(f'{val}/{year}', format='%m/%d/%Y'))
        except:
            parsed.append(pd.NaT)
    return pd.DatetimeIndex(parsed)


def process_file(filepath, year):
    """Read a single xlsx and return a clean dataframe: dates as index, cities as columns."""
    df = pd.read_excel(filepath, header=None)

    if year in ['2015', '2016']:
        dates   = parse_dates(df.iloc[0, 1:].tolist(), year)
        df_data = df.iloc[1:, :].set_index(df.columns[0])
    else:
        dates   = parse_dates(df.iloc[2, 1:].tolist(), year)
        df_data = df.iloc[3:, :].set_index(df.columns[0])

    df_data.columns = dates

    # Transpose: dates → index, cities → columns
    df_t = df_data.T
    df_t.index.name = 'date'

    # Clean city names
    df_t.columns = df_t.columns.astype(str).str.replace('*', '', regex=False).str.strip()

    # Drop unparseable dates and coerce values to numeric
    df_t = df_t[df_t.index.notna()]
    df_t = df_t.apply(pd.to_numeric, errors='coerce')

    return df_t

## Step 2 — Load all files

In [ ]:
all_years = {}

for filepath in sorted(raw_folder.glob('*.xlsx')):
    year = filepath.stem.split('_')[-1]
    try:
        df_t = process_file(filepath, year)
        all_years[year] = df_t
        print(f'✅ {year} — shape: {df_t.shape}, dates: {df_t.index[0].date()} → {df_t.index[-1].date()}')
    except Exception as e:
        print(f'❌ {year} — {e}')

## Step 3 — Combine and clean

In [ ]:
df_combined = pd.concat(all_years.values()).sort_index()
df_combined = df_combined[~df_combined.index.duplicated(keep='first')]

# Compute PEEL REGION before dropping neighborhoods
peel_cols = [c for c in ['BRAMPTON', 'MISSISSAUGA'] if c in df_combined.columns]
if peel_cols:
    df_combined['PEEL REGION'] = df_combined[peel_cols].mean(axis=1)

# Drop junk and GTA neighborhood columns
DROP_COLS = [
    'nan', 'Average', 'S-Simple V-Volume Weighted P-Population Weighted',
    'NORTH YORK', 'SCARBOROUGH', 'ETOBICOKE', 'BRAMPTON',
    'MISSISSAUGA', 'SARNIA', 'VAUGHAN/MARKHAM'
]
df_combined = df_combined.drop(
    columns=[c for c in df_combined.columns if str(c) in DROP_COLS],
    errors='ignore'
)

# Standardize city and aggregate column names across old/new formats
RENAME = {
    'SAINTJOHN'                 : 'SAINT JOHN',
    'STJOHNS'                   : 'ST JOHNS',
    'TORONTO'                   : 'CITY OF TORONTO',
    'TROIS RIVIÉRES'            : 'TROIS RIVIÈRES',
    'CANADA (S)'                : 'Canada(S)',
    'CANADA (P)'                : 'Canada(P)',
    'Canada Ave (V)'            : 'Canada Ave(V)',
    'Western Average (P)'       : 'Western Ave(P)',
    'Ontario Average (P)'       : 'Ontario Ave(P)',
    'Québec Average (P)'        : 'Quebec Ave(P)',
    'Atlantic Average (P)'      : 'Atlantic Ave(P)',
    'Large Markets Average (S)' : 'Large Markets Ave(S)',
    'Large Markets Average (P)' : 'Large Markets Ave(P)',
    'Small Markets Average (S)' : 'Small Markets Ave(S)',
    'Small Markets Average (P)' : 'Small Markets Ave(P)',
}
df_combined = df_combined.rename(columns=RENAME)

# Merge any remaining duplicate column names
df_combined = df_combined.T.groupby(level=0).first().T

print(f'Shape     : {df_combined.shape}')
print(f'Date range: {df_combined.index.min().date()} → {df_combined.index.max().date()}')
print(f'Columns   : {len(df_combined.columns)}')
df_combined.head(3)

## Step 4 — Save

In [ ]:
output_path = processed_folder / 'gas_prices_all_years.csv'
df_combined.to_csv(output_path)
print(f'✅ Saved to {output_path}')
print(f'   Shape     : {df_combined.shape}')
print(f'   Date range: {df_combined.index.min().date()} → {df_combined.index.max().date()}')